# Assignment 2 Stage 2: Hidden Test Evaluation

For this part I'm just loading the exact model I already saved in Stage 1 (model_checkpoint/vectorizer.joblib and model.joblib) and using it to predict on the hidden test set. I'm not retraining or changing the model at all here, just running it on new data, since that's what the assignment asks for.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## Load the Stage 1 Checkpoint (No Retraining)

In [ ]:
vectorizer = joblib.load('model_checkpoint/vectorizer.joblib')
model = joblib.load('model_checkpoint/model.joblib')
sia = SentimentIntensityAnalyzer()

def vader_features(texts):
    rows = [sia.polarity_scores(t) for t in texts]
    return np.array([[r['neg'], r['neu'], r['pos'], r['compound']] for r in rows])

def build_features(texts, vectorizer):
    tfidf = vectorizer.transform(texts)
    vfeat = csr_matrix(vader_features(texts))
    return hstack([tfidf, vfeat]).tocsr()

print(type(model).__name__, 'loaded from checkpoint.')

## Load the Hidden Test Set and Run Inference 

In [ ]:
hidden = pd.read_csv('hidden_test_with_labels.csv')
print('Hidden test shape:', hidden.shape)
print(hidden['label_name'].value_counts())

X_hidden = build_features(hidden['text'].values, vectorizer)
pred_hidden = model.predict(X_hidden)

## Hidden Test Accuracy and Confusion Matrix

In [ ]:
y_hidden = hidden['label'].values
acc_hidden = accuracy_score(y_hidden, pred_hidden)
f1_hidden = f1_score(y_hidden, pred_hidden, average='macro')

print(f'Hidden test accuracy: {acc_hidden:.4f}')
print(f'Hidden test macro F1: {f1_hidden:.4f}')
print(classification_report(y_hidden, pred_hidden, target_names=['negative', 'positive']))

cm_hidden = confusion_matrix(y_hidden, pred_hidden)
print(cm_hidden)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm_hidden, display_labels=['negative', 'positive']).plot(ax=ax, cmap='Purples', colorbar=False)
ax.set_title(f'Hidden Test Confusion Matrix\nacc={acc_hidden:.3f}, macroF1={f1_hidden:.3f}')
plt.tight_layout()
plt.show()

Hidden test result: accuracy = 0.6133 (61.3%), macro-F1 = 0.5943.

Confusion matrix:
[[119 181]
[ 51 249]]

## Public Test VS Hidden Test Comparison

In [ ]:
# Public test numbers are copied from stage1_notebook.ipynb (same checkpoint, no retraining)
public_acc, public_f1 = 0.6250, 0.6110
public_cm = np.array([[87, 113], [37, 163]])

comparison = pd.DataFrame({
    'accuracy': [public_acc, acc_hidden],
    'macro_f1': [public_f1, f1_hidden],
    'n_examples': [400, len(hidden)],
}, index=['public_test', 'hidden_test'])
print(comparison)

Comparing this to my public test result from Stage 1 (62.5% accuracy, macro F1 0.611), the hidden test numbers are pretty close, just a little bit lower, about 1.2 points less accuracy and 0.017 less macro F1. Looking at both confusion matrices, I'm seeing the same pattern both
times: the model is good at catching positive reviews (recall around 0.81-0.83) but worse at catching negative ones (recall only 0.40-0.43), so it tends to guess "positive" too often. This makes sense given I only had 60 negative reviews out of 240 to train on, so the model just didn't get to see enough different ways people write negative reviews.

I think it's actually a good sign that the hidden test score is so close to my public test score instead of dropping a lot. That tells me the model is generalizing consistently to data it's never seen before, rather than just being tuned to whatever happened to be in public_test.csv. The small drop makes sense too since the hidden set is truly new data I never looked at while building the model, unlike public_test.csv which I checked a bunch of times during development.

## Save Hidden Test Predictions

In [ ]:
out = pd.DataFrame({'id': hidden['id'], 'predicted_label': pred_hidden})
out.to_csv('hidden_test_predictions.csv', index=False)
out.head()

## If I Had More Time or More Compute

- Fine-tune a pretrained transformer (e.g. DistilBERT or RoBERTa). This wasn't possible in my Stage 1 development environment since it required downloading pretrained weights, which wasn't reachable there. A pretrained tokenizer would directly address the ~48% out-of-vocabulary rate The one measured in Stage 1, and pretrained language understanding would likely generalize far better than a bag-of-words model trained on only 240 reviews.
- Get more negative training examples, even a modest amount (e.g. 120-150 instead of 60). Every diagnostic in this assignment,  the CV vs public test gap, the public vs hidden test gap, the consistently low negative recall - points to the same root cause: too few negative examples,  not a weakness in model architecture.
- Try data augmentation for the minority class (e.g. back-translation or synonym replacement on the 60 negative reviews) instead of plain duplication, so the model sees more varied negative phrasing rather than memorizing the same reviews' exact wording multiple times.
- Combine multiple sentiment lexicons (e.g. VADER + SentiWordNet) as additional features, to further reduce reliance on the small training vocabulary.